# Pipeline Parallel Advanced: Scheduling and Optimization

## Overview

Advanced pipeline parallelism techniques including scheduling algorithms and bubble optimization.

### Topics Covered
- GPipe vs 1F1B scheduling
- Interleaved pipeline
- Bubble analysis
- Memory optimization

## 1. Scheduling Comparison

### GPipe Schedule
```
Time →
GPU0: [F0][F1][F2][F3]          [B3][B2][B1][B0]
GPU1:    [F0][F1][F2][F3]    [B3][B2][B1][B0]
GPU2:       [F0][F1][F2][F3][B3][B2][B1][B0]
GPU3:          [F0][F1][F2][F3][B3][B2][B1][B0]

Memory: O(num_microbatches) - stores all activations
```

### 1F1B Schedule
```
Time →
GPU0: [F0][F1][F2][F3][B0][F4][B1][F5][B2][B3]
GPU1:    [F0][F1][F2][B0][F3][B1][F4][B2][B3]
GPU2:       [F0][F1][B0][F2][B1][F3][B2][B3]
GPU3:          [F0][B0][F1][B1][F2][B2][F3][B3]

Memory: O(num_stages) - constant memory
```

In [ ]:
def calculate_bubble_ratio(num_stages, num_microbatches):
    """Calculate pipeline bubble ratio.
    
    Bubble ratio = (p-1) / m where p=stages, m=microbatches
    """
    bubble = (num_stages - 1) / num_microbatches
    efficiency = 1 - bubble
    
    print(f"Stages: {num_stages}, Microbatches: {num_microbatches}")
    print(f"Bubble ratio: {bubble:.1%}")
    print(f"Pipeline efficiency: {efficiency:.1%}")
    return bubble

# Compare different configurations
calculate_bubble_ratio(4, 8)   # 37.5% bubble
calculate_bubble_ratio(4, 16)  # 18.75% bubble
calculate_bubble_ratio(4, 32)  # 9.4% bubble

## 2. Interleaved Pipeline

Assign multiple non-consecutive layers to each GPU to reduce bubble.

In [ ]:
def interleaved_assignment(num_layers, num_stages, num_chunks=2):
    """Assign layers with interleaving.
    
    Example: 24 layers, 4 stages, 2 chunks
    GPU0: layers 0-2, 12-14
    GPU1: layers 3-5, 15-17
    GPU2: layers 6-8, 18-20
    GPU3: layers 9-11, 21-23
    """
    layers_per_stage = num_layers // (num_stages * num_chunks)
    
    assignment = {i: [] for i in range(num_stages)}
    
    for chunk in range(num_chunks):
        for stage in range(num_stages):
            start = (chunk * num_stages + stage) * layers_per_stage
            end = start + layers_per_stage
            assignment[stage].extend(range(start, end))
    
    print(f"Interleaved assignment ({num_layers} layers, {num_stages} stages, {num_chunks} chunks):")
    for stage, layers in assignment.items():
        print(f"  GPU {stage}: layers {layers}")
    
    return assignment

interleaved_assignment(24, 4, 2)

## 3. Summary

| Schedule | Memory | Bubble | Complexity |
|----------|--------|--------|------------|
| GPipe | O(m) | (p-1)/m | Simple |
| 1F1B | O(p) | (p-1)/m | Medium |
| Interleaved | O(p) | (p-1)/(m*v) | Complex |